# SupeRR Selector — player ratings

This notebook reads the silver files and builds two lists: batters and bowlers.

Each player has a baseline from their usual rates, then three pressure bands, a within-match comparison, a clutch term, form, and consistency. A season they missed counts as zero. Every career rate is divided by the same five-season total, 4.0951. The weights are 0.6561, 0.729, 0.81, 0.9, and 1 for 2022 through 2026.

The calculations live in `player_ratings.py`. This notebook runs them and shows the tables. The silver files are not changed.

## Load and tag balls

A legal ball is not a wide, a no-ball, or a super over. A wide or a no-ball adds one conceded run. Bye and leg-bye amounts are not in the file, and neither are extra runs beyond that one-run penalty, so a team total is still a lower bound. An ordinary chase target is the first-innings total plus one. A DLS target is used as supplied.

A legal ball is not a wide, a no-ball, or a super over. Situation points come from the phase of the innings, wickets down, balls left in a chase, required rate, a close chase, a knockout, a final, and rain. Zero is 0–2 points, medium is 3–5, and high is 6 or more.

In [1]:
from pathlib import Path

import pandas as pd

import player_ratings as ratings

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 30)

balls, matches, comps = ratings.load()
work, legal = ratings.add_situation(balls)
print("deliveries", len(balls), "legal, excluding super overs", len(legal))
print("five-season weight total", round(ratings.W_SUM, 4))
bands = ratings.band_summary(legal)
display(bands.assign(share=lambda frame: (100 * frame["share"]).round(1)))

deliveries 200999 legal, excluding super overs 192675
five-season weight total 4.0951


,balls,share
band,,
zero,129349,67.1
medium,51104,26.5
high,12222,6.3


## Elo

Team strength updates after each match. Batting strength and bowling strength update after each legal ball, on separate IPL and international ladders. A player with few balls is held near 1500, and that hold loosens as the career lengthens. Venue strength uses only balls already bowled at that ground. A ball is worth the runs and the wicket chance minus what those ratings expected.

In [2]:
mu, p0, rho = ratings.expectation_tables(legal)
legal = ratings.fit_elo(legal, mu, p0)
summary = legal.attrs["elo_summary"]
for key, value in summary.items():
    if isinstance(value, str):
        print(key + ":", value)
    else:
        print(key + ":", round(value, 1))
print("mean bat runs per legal ball", round(rho, 3))

k_run: 4.0
k_wicket: 16.0
k_team: 20.0
batter_elo_min: 917.5
batter_elo_max: 2566.0
batter_elo_sd: 146.8
bowler_elo_min: 1019.6
bowler_elo_max: 2125.4
bowler_elo_sd: 125.8
team_elo_min: 1361.3
team_elo_max: 1647.0
grounds: 97
step_sizes: as written
mean bat runs per legal ball 1.394


## Batter scores

Baseline is strike rate and average against the phases that batter actually plays. The three bands are the strength-adjusted ball value inside zero, medium, and high pressure. Peers compare an innings with teammates and the opposition. Clutch is that comparison in the high band. Form uses up to eight earlier innings. Consistency prefers a narrower spread of match values.

Each term is a z-score on players with at least 30 decay-weighted legal balls, and each z-score stops at 3. The composite is 0.20 baseline, 0.10 zero, 0.15 medium, 0.20 high, 0.15 clutch, 0.10 peers, 0.05 form, and 0.05 consistency. The extra 0.05, which the eight original shares did not account for, is on the baseline.

In [3]:
batters = ratings.score_batters(legal)
print("batters", len(batters), "ranked", int(batters["ranked"].sum()))
batter_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "runs", "strike_rate", "average", "n_eff", "high_n_eff",
]
display(ratings.top_table(batters, batter_cols))
print("roles in the batter top 10")
print(batters.loc[batters["rank"].le(10), "role"].value_counts().to_string())

batters 888 ranked 70


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,runs,strike_rate,average,n_eff,high_n_eff
0,1.0,Tim David,middle order,1.320,2.211,0.691,1.568,0.934,2.095,0.869,0.623,-0.908,2153,177.202,34.726,1002.709,108.700
1,2.0,Naman Dhir,top order,0.922,1.372,-0.252,1.350,2.296,-0.287,1.378,0.395,-2.068,992,168.707,28.343,551.980,54.880
2,3.0,David Miller,middle order,0.916,1.585,-0.943,0.458,1.437,3.000,-0.245,-1.276,-0.495,1902,149.060,38.816,977.315,86.447
3,4.0,Mohammad Nawaz,lower order,0.890,0.094,-0.492,1.145,0.594,3.000,1.309,1.448,-0.463,705,142.424,21.364,405.336,72.205
4,5.0,Jos Buttler,top order,0.880,2.064,0.879,0.957,0.671,-0.224,1.523,-0.122,-0.223,4384,155.461,41.752,2303.778,37.050
5,6.0,Rinku Singh,middle order,0.833,1.689,-0.910,0.748,2.981,-0.224,0.658,-1.527,-1.558,1949,152.623,38.980,1047.970,99.733
6,7.0,Rovman Powell,middle order,0.795,0.786,-0.267,0.444,1.014,3.000,0.214,-0.127,-1.395,2287,147.074,28.235,1242.328,143.941
7,8.0,Dinesh Karthik,middle order,0.652,1.482,-0.827,0.765,0.357,1.595,1.264,0.050,-2.327,1061,166.562,31.206,452.142,62.200
8,9.0,Heinrich Klaasen,middle order,0.596,2.315,1.445,0.666,-0.756,-0.264,1.267,-0.932,-0.016,2356,167.092,39.932,1199.960,36.698
9,10.0,Mahendra Singh Dhoni,lower order,0.560,0.923,0.090,0.581,1.507,-0.224,1.267,0.101,-2.412,690,150.000,31.364,352.416,125.093


roles in the batter top 10
role
middle order    6
top order       2
lower order     2


## Bowler scores

The same shape. Baseline uses economy and balls per wicket, against the usual rate for that bowler's pace or spin and the phases they bowl. A lower rate is a higher score. The high band, clutch, and peers use the same direction.

In [4]:
bowlers = ratings.score_bowlers(legal, work)
print("bowlers", len(bowlers), "ranked", int(bowlers["ranked"].sum()))
bowler_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "wickets", "economy", "balls_per_wicket", "n_eff", "high_n_eff",
]
display(ratings.top_table(bowlers, bowler_cols))
print("roles in the bowler top 10")
print(bowlers.loc[bowlers["rank"].le(10), "role"].value_counts().to_string())

bowlers 647 ranked 102


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,wickets,economy,balls_per_wicket,n_eff,high_n_eff
0,1.0,Jasprit Bumrah,Pace,1.507,1.256,1.913,1.575,1.441,1.421,2.867,0.841,-0.037,85.0,6.920,20.106,1455.083,126.834
1,2.0,Ahmed Daniyal,Pace,1.453,1.590,1.522,1.439,3.000,-0.392,1.881,-0.002,0.761,10.0,5.018,17.100,152.010,44.730
2,3.0,Sunil Narine,Spin,1.349,0.441,0.376,1.168,3.000,0.581,2.842,0.254,1.280,64.0,6.773,23.906,1246.702,33.455
3,4.0,Abrar Ahmed,Spin,1.146,1.332,0.198,1.432,1.461,1.418,1.525,-0.349,0.090,46.0,6.565,16.609,689.250,55.500
4,5.0,Rashid Khan,Spin,1.142,0.691,0.145,0.794,1.240,2.889,1.542,0.684,0.008,153.0,7.332,17.954,2235.578,123.312
5,6.0,Sufiyan Muqeem,Spin,1.128,1.942,-0.756,1.590,0.498,2.254,1.705,0.630,-1.269,39.0,5.919,13.282,449.880,57.600
6,7.0,Jofra Archer,Pace,1.077,0.267,0.490,0.740,3.000,0.569,0.998,0.969,0.598,71.0,8.636,17.761,1144.803,76.034
7,8.0,Jamie Overton,Pace,0.985,0.898,0.903,1.626,1.714,0.882,0.413,0.657,-1.568,36.0,8.253,13.833,470.300,43.200
8,9.0,Krunal Pandya,Spin,0.949,0.010,-0.171,0.560,3.000,1.032,1.359,-0.108,-0.091,56.0,7.628,22.446,1045.079,53.557
9,10.0,Mitchell Starc,Pace,0.942,0.163,0.272,1.457,1.423,3.000,-0.277,-0.758,-0.110,53.0,9.194,15.170,672.678,50.323


roles in the bowler top 10
role
Pace    5
Spin    5


## Rankings

A player is ranked in a skill with at least 120 decay-weighted legal balls, and at least 30 of those in the high band. Anyone under the line stays in the full table with no rank. The two lists are saved under `ratings/`.

In [5]:
out = Path("ratings")
out.mkdir(exist_ok=True)
batters.to_csv(out / "batters.csv", index=False)
bowlers.to_csv(out / "bowlers.csv", index=False)
print("wrote", out / "batters.csv", "and", out / "bowlers.csv")
print("decision: floor is 120 decay-weighted legal balls and 30 high-band balls")
print("decision: season denominator", round(ratings.W_SUM, 4))
print("decision: elo step sizes", summary["step_sizes"])

wrote ratings\batters.csv and ratings\bowlers.csv
decision: floor is 120 decay-weighted legal balls and 30 high-band balls
decision: season denominator 4.0951
decision: elo step sizes as written
